# Initialize Dataset

In [2]:
# ============================================================
# PQD-16c CNN-RES-BiLSTM V2
# DATASET INITIALIZATION
# ============================================================
import pandas as pd
import numpy as np
import torch
from pathlib import Path

DATA_DIR = Path("../dataset/PQD-16c-Data")

TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV   = DATA_DIR / "validation.csv"
TEST_CSV  = DATA_DIR / "test.csv"

# Load CSV files
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Columns
LABEL_COLUMN = train_df.columns[0]
SIGNAL_COLUMNS = train_df.columns[1:]

# Signals
X_train = train_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_val   = val_df[SIGNAL_COLUMNS].values.astype(np.float32)
X_test  = test_df[SIGNAL_COLUMNS].values.astype(np.float32)

# Labels
y_train_raw = train_df[LABEL_COLUMN].values
y_val_raw   = val_df[LABEL_COLUMN].values
y_test_raw  = test_df[LABEL_COLUMN].values

# Class mapping
class_names = sorted(np.unique(y_train_raw))
label_to_index = {
    label: i for i, label in enumerate(class_names)
}

y_train = np.array(
    [label_to_index[x] for x in y_train_raw],
    dtype=np.int64
)

y_val = np.array(
    [label_to_index[x] for x in y_val_raw],
    dtype=np.int64
)

y_test = np.array(
    [label_to_index[x] for x in y_test_raw],
    dtype=np.int64
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

print("=" * 70)
print("PQD-16c DATASET INITIALIZED")
print("=" * 70)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

print("\nClasses:")
for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name}")

print("=" * 70)

PQD-16c DATASET INITIALIZED
Train: torch.Size([1024, 1, 640]) torch.Size([1024])
Val  : torch.Size([256, 1, 640]) torch.Size([256])
Test : torch.Size([320, 1, 640]) torch.Size([320])

Classes:
 0 -> Flicker
 1 -> Flicker+Harmonics
 2 -> Flicker+Sag
 3 -> Flicker+Swell
 4 -> Harmonics
 5 -> Impulsive Transient
 6 -> Interruption
 7 -> Interruption+Harmonics
 8 -> Normal
 9 -> Notch
10 -> Oscillatory transient
11 -> Sag
12 -> Sag+Harmonics
13 -> Spike
14 -> Swell
15 -> Swell+Harmonics


# Predict a sample

In [27]:
# ============================================================
# PQD-16c CNN-RES-BiLSTM V2
# MODEL + SINGLE TEST SAMPLE PREDICTION
# ============================================================

import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = Path("../models/legacy/cnn_res_bilstm_pq.pth")


# ============================================================
# RESIDUAL CNN BLOCK
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, dilation=1):
        super().__init__()
        padding = ((kernel_size - 1) * dilation) // 2

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.act1 = nn.GELU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.skip = (
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm1d(out_channels)
            )
            if in_channels != out_channels else nn.Identity()
        )

        self.activation = nn.GELU()

    def forward(self, x):
        identity = self.skip(x)
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.activation(x + identity)


# ============================================================
# CNN-RES-BiLSTM MODEL
# ============================================================

class CNNResBiLSTM(nn.Module):
    def __init__(self, num_classes=16):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.GELU(),

            ResidualConvBlock(32, 32, 5, 1),
            nn.MaxPool1d(2),

            ResidualConvBlock(32, 64, 5, 1),
            nn.MaxPool1d(2),

            ResidualConvBlock(64, 128, 5, 2),
            nn.MaxPool1d(2),

            ResidualConvBlock(128, 128, 3, 2)
        )

        self.sequence_norm = nn.LayerNorm(128)

        self.lstm = nn.LSTM(
            128, 128,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True
        )

        self.attention = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.ndim == 4:
            x = x.squeeze(2)

        x = self.cnn(x).transpose(1, 2)
        x = self.sequence_norm(x)
        x, _ = self.lstm(x)

        weights = torch.softmax(self.attention(x), dim=1)
        x = torch.sum(x * weights, dim=1)

        return self.classifier(x)


# ============================================================
# LOAD MODEL
# ============================================================

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = CNNResBiLSTM(
    checkpoint["num_classes"]
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


# ============================================================
# PREDICTION
# ============================================================

print("=" * 70)
print("PQD-16c CNN-RES-BiLSTM V2 PREDICTION")
print("=" * 70)
print("Device     :", DEVICE)
print("Model path :", MODEL_PATH.resolve())
print("=" * 70)

while True:
    try:
        sample_number = int(
            input("\nEnter test sample number (1-320): ")
        )

        if 1 <= sample_number <= len(X_test):
            break

        print(
            f"Invalid input. Enter a number from 1 to {len(X_test)}."
        )

    except ValueError:
        print("Please enter a valid integer.")

index = sample_number - 1

sample = X_test[index:index + 1].to(DEVICE)

with torch.no_grad():
    outputs = model(sample)
    probabilities = torch.softmax(outputs, dim=1)

    predicted_index = outputs.argmax(dim=1).item()
    confidence = probabilities[0, predicted_index].item() * 100

actual_index = y_test[index].item()


# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 70)
print("PREDICTION RESULT")
print("=" * 70)
print(f"Test Sample : {sample_number}")
print(f"Actual      : {class_names[actual_index]}")
print(f"Predicted   : {class_names[predicted_index]}")
print(f"Confidence  : {confidence:.2f}%")
print("=" * 70)

PQD-16c CNN-RES-BiLSTM V2 PREDICTION
Device     : cpu
Model path : /mnt/ntfs/my_docs/VIT_Y4_SEM7/C1_XAI/project/models/cnn_res_bilstm_pq.pth



Enter test sample number (1-320):  99



PREDICTION RESULT
Test Sample : 99
Actual      : Harmonics
Predicted   : Sag+Harmonics
Confidence  : 77.34%
